In [ ]:
!pip install google-genai langchain langchain-community langchain-text-splitters pypdf qdrant-client rank-bm25 sentence-transformers transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# =====================================================================
# CELL 1: Imports and Setup
# =====================================================================
import uuid
import numpy as np
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

# Initialize Local Models (Replace with specific local models as needed)
# Using a fast embedding model for Qdrant
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
# Cross-encoder for reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# Local LLM for semantic tagging and final generation (e.g., Llama-3-8B or Mistral)
# Quantized models via bitsandbytes or vLLM are recommended for local notebook execution
llm_pipeline = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta", device_map="auto")

/tmp/ipykernel_5528/3797324785.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
import os
import re
import time
from typing import Any, List, Dict, Optional

# Fallback dummy structures in case external dependencies are missing
try:
    from google import genai
except ImportError:
    genai = None

class DummyFailoverManager:
    def __init__(self, primary_model, fallback_model):
        self.primary_model = primary_model
        self.fallback_model = fallback_model
    def generate_content(self, client, contents, config):
        # Safe fallback generation logic if ModelFailoverManager is not defined
        try:
            res = client.models.generate_content(model=self.primary_model, contents=contents, config=config)
            return res, self.primary_model
        except Exception as e:
            res = client.models.generate_content(model=self.fallback_model, contents=contents, config=config)
            return res, self.fallback_model

try:
    from model_utils import ModelFailoverManager
except ImportError:
    ModelFailoverManager = DummyFailoverManager

def extract_response_text(response) -> str:
    try:
        if response and hasattr(response, "text") and response.text is not None:
            return response.text
        return ""
    except Exception:
        return ""

DEFAULT_MODEL = "gemma-4-31b-it"
FALLBACK_MODEL = "gemma-4-26b-a4b-it"

class GemmaGenAIModel:
    """
    Model adapter connecting Google GenAI SDK (default: gemma-4-31b-it with failover
    to gemma-4-26b-a4b-it on request exhaustion) to the mini-swe-agent Model protocol.
    """

    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        fallback_model: str = FALLBACK_MODEL,
        client: Any = None,
        api_key: Optional[str] = None
    ):
        self.model_name = model_name
        self.fallback_model = fallback_model
        self.client = client
        if self.client is None:
            key = api_key or os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
            if key:
                try:
                    from google import genai
                    self.client = genai.Client(api_key=key)
                except ImportError:
                    self.client = None

        self.failover_manager = ModelFailoverManager(
            primary_model=self.model_name,
            fallback_model=self.fallback_model,
        )
        self.action_regex = r"```(?:mswea_bash_command|bash)?\s*\n(.*?)\n```"
        self.cost = 0.0
        self.n_calls = 0

    def query(self, messages: List[Dict[str, Any]], **kwargs) -> Dict[str, Any]:
        """Query Gemma model (with quota-exhaustion failover) and parse actions."""
        if not self.client:
            raise RuntimeError(
                "google-genai client not available. Ensure 'google-genai' is installed "
                "and GOOGLE_API_KEY is set or client is provided."
            )

        from google.genai import types

        prompt_parts = []
        for msg in messages:
            role = msg.get("role", "user").upper()
            content = msg.get("content", "")
            prompt_parts.append(f"[{role}]\n{content}\n")

        full_prompt = "\n".join(prompt_parts)

        self.n_calls += 1
        response, used_model = self.failover_manager.generate_content(
            client=self.client,
            contents=full_prompt,
            config=types.GenerateContentConfig(
                temperature=kwargs.get("temperature", 0.6),
                max_output_tokens=kwargs.get("max_output_tokens", 16384),
            ),
        )
        self.model_name = used_model

        text = extract_response_text(response)
        actions = self._parse_actions(text)

        return {
            "role": "assistant",
            "content": text,
            "extra": {
                "actions": actions,
                "cost": 0.0,
                "model_used": used_model,
                "timestamp": time.time(),
            },
        }

    def _parse_actions(self, content: str) -> List[Dict[str, str]]:
        if not content:
            return []
        matches = re.findall(self.action_regex, content, re.DOTALL)
        if not matches:
            for line in content.splitlines():
                line_str = line.strip()
                if line_str.startswith("python edit_harness.py") or line_str.startswith("./edit_harness.py"):
                    return [{"command": line_str}]
            return []
        return [{"command": matches[0].strip()}]

    def format_message(self, **kwargs) -> Dict[str, Any]:
        return kwargs

In [ ]:
# =====================================================================
# CELL 2: Document Processing & Chunking
# =====================================================================
def load_and_chunk_documents(file_paths: List[str]) -> List[Dict[str, Any]]:
    """
    Loads PDFs, extracts text with page numbers, and splits into chunks.
    Respects section boundaries by prioritizing double-newlines.
    Target: ~400 words per chunk with 10% (40 words) overlap.
    """
    # Using roughly 1 word ≈ 1.25 tokens. 400 words = ~500 chars if we scale,
    # but let's use exact word counts via a custom length function or standard char counts.
    # Standard English word is ~5 chars + 1 space = 6 chars.
    # 400 words * 6 chars = 2400 chars. 10% overlap = 240 chars.
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n## ", "\n\n", "\n", ".", " "], # Respects section boundaries
        chunk_size=2400,
        chunk_overlap=240,
        length_function=len
    )

    chunks = []
    for path in file_paths:
        loader = PyPDFLoader(path)
        pages = loader.load()

        for page in pages:
            page_chunks = text_splitter.split_text(page.page_content)
            for chunk in page_chunks:
                chunks.append({
                    "id": str(uuid.uuid4()),
                    "text": chunk,
                    "metadata": {
                        "source": path,
                        "page": page.metadata.get("page", 0) + 1
                    }
                })
    return chunks

In [ ]:
# =====================================================================
# CELL 3: Semantic Labeling (Context Tagging) with GemmaGenAIModel
# =====================================================================
def tag_chunks_with_llm(chunks: List[Dict[str, Any]], model_client: GemmaGenAIModel) -> List[Dict[str, Any]]:
    """
    Passes each chunk to the Gemma GenAI model to generate a 1-2 sentence description.
    This description is prepended or stored as global context for the chunk.
    """
    tagged_chunks = []
    for chunk in chunks:
        # Construct prompt asking for a brief description
        system_prompt = "You are a helpful assistant. Provide a 1-2 sentence semantic description of the following text."
        user_prompt = f"Source Document: {chunk['metadata']['source']}, Page {chunk['metadata']['page']}\n\nText:\n{chunk['text']}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        # Generate description using Gemma GenAI Model
        response = model_client.query(messages, temperature=0.3, max_output_tokens=100)
        description = response["content"].strip()

        # Store description in metadata and prepend to the text for embedding
        chunk["metadata"]["semantic_description"] = description
        chunk["enriched_text"] = f"Context: {description}\n\nContent: {chunk['text']}"
        tagged_chunks.append(chunk)

    return tagged_chunks

In [ ]:
# =====================================================================
# CELL 4: Indexing (Vector DB & BM25)
# =====================================================================
class HybridRetriever:
    def __init__(self, chunks: List[Dict[str, Any]]):
        self.chunks = chunks
        self.qdrant = QdrantClient(":memory:") # Using in-memory Qdrant for notebook prototype

        # 1. Setup Vector Store
        self.qdrant.create_collection(
            collection_name="tutorial_docs",
            vectors_config=VectorParams(size=384, distance=Distance.COSINE) # size matches all-MiniLM-L6-v2
        )

        points = []
        for chunk in chunks:
            vector = embedding_model.encode(chunk["enriched_text"]).tolist()
            points.append(PointStruct(
                id=chunk["id"],
                vector=vector,
                payload={"text": chunk["text"], "metadata": chunk["metadata"]}
            ))
        self.qdrant.upsert(collection_name="tutorial_docs", points=points)

        # 2. Setup BM25
        # Tokenize enriched text for exact keyword matching
        tokenized_corpus = [chunk["enriched_text"].lower().split() for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, top_k: int = 150) -> List[Dict[str, Any]]:
        # --- Vector Search ---
        query_vector = embedding_model.encode(query).tolist()
        vector_results = self.qdrant.query_points(
            collection_name="tutorial_docs",
            query_vector=query_vector,
            limit=top_k
        )
        vector_ranked = {res.id: (rank, res.payload) for rank, res in enumerate(vector_results)}

        # --- BM25 Search ---
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        # Get top K indices sorted by score descending
        bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k]
        bm25_ranked = {self.chunks[i]["id"]: (rank, self.chunks[i]) for rank, i in enumerate(bm25_top_indices)}

        # --- Reciprocal Rank Fusion (RRF) ---c
        rrf_scores = {}
        k_constant = 60

        all_ids = set(vector_ranked.keys()).union(set(bm25_ranked.keys()))
        for doc_id in all_ids:
            rrf_score = 0.0
            if doc_id in vector_ranked:
                rank = vector_ranked[doc_id][0] + 1
                rrf_score += 1.0 / (k_constant + rank)
            if doc_id in bm25_ranked:
                rank = bm25_ranked[doc_id][0] + 1
                rrf_score += 1.0 / (k_constant + rank)

            # Grab payload from either dictionary
            payload = vector_ranked[doc_id][1] if doc_id in vector_ranked else bm25_ranked[doc_id][1]
            rrf_scores[doc_id] = {"score": rrf_score, "payload": payload}

        # Sort by RRF score descending and return top 150 combined
        fused_results = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]
        return [item["payload"] for item in fused_results]

In [ ]:
# =====================================================================
# CELL 5: Reranking & Final Gemma GenAI Generation
# =====================================================================
def answer_query(query: str, retriever: HybridRetriever, model_client: GemmaGenAIModel) -> str:
    # 1. Retrieve top 150 via Hybrid Search
    top_150 = retriever.retrieve(query, top_k=150)

    # 2. Cross-Encoder Reranking
    # Prepare pairs of (Query, Document)
    pairs = [[query, doc["text"]] for doc in top_150]
    rerank_scores = cross_encoder.predict(pairs)

    # Sort documents by cross-encoder score
    ranked_indices = np.argsort(rerank_scores)[::-1]
    top_5_docs = [top_150[i] for i in ranked_indices[:5]]

    # 3. Context Construction for Gemma LLM
    context_str = ""
    for i, doc in enumerate(top_5_docs):
        source = doc["metadata"]["source"]
        page = doc["metadata"]["page"]
        context_str += f"[Source: {source}, Page: {page}]\n{doc['text']}\n\n"

    # 4. Final Gemma Generation
    system_prompt = (
        "You are a scholarly assistant. Answer the user's question using ONLY the provided context. "
        "You MUST cite the source document and page number inline for every factual claim you make "
        "(e.g., [Cite: PC141labRules.pdf, Page: 2]). If the answer is not in the context, state that you do not know."
    )
    user_prompt = f"Context:\n{context_str}\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    response = model_client.query(messages, temperature=0.3, max_output_tokens=512)
    return response["content"].strip()

In [ ]:
[
  {
    "reference_file": "PC141labRAG.pdf[cite: 1]",
    "question": "What does a difference of a full order of magnitude typically signal when making an estimate?[cite: 1]",
    "answer_location": "Section 3.1 (Estimation), Page 2, specifically under the 'Mini rule-of-thumb'[cite: 1]."
  },
  {
    "reference_file": "PC141labRAG.pdf[cite: 1]",
    "question": "How is the range of possible values defined when establishing upper and lower bounds?[cite: 1]",
    "answer_location": "Section 3.2.2 (Range of Possible Values), Page 2[cite: 1]."
  },
  {
    "reference_file": "PC141labRAG.pdf[cite: 1]",
    "question": "If two numerical estimates fall within the same cycle on a logarithmic scale, what does this indicate about their relationship?[cite: 1]",
    "answer_location": "Section 3.6 (Logarithmic Scale), Page 10, designated under the 'Key idea'[cite: 1]."
  },
  {
    "reference_file": "PC141labRAG.pdf[cite: 1]",
    "question": "How should a student adjust their uncertainty if their best estimate does not fall exactly midway between their upper and lower bounds?[cite: 1]",
    "answer_location": "Section 3.7 (Uncertainties), Page 13, under step 2 of the 'IT4' instructions[cite: 1]."
  },
  {
    "reference_file": "PC141labRAG.pdf[cite: 1]",
    "question": "When utilizing scientific notation, why is it incorrect to write a value as 12 x 10^3?[cite: 1]",
    "answer_location": "Appendix A.2 (Scientific Notation), Page 18, under 'Common pitfalls'[cite: 1]."
  }
]

[
  {
    "question": "Who is the instructor for the course, and what is their email address?[cite: 1]",
    "location_of_answer": "At the very top of the document in the header section, just above the 'Description' heading.[cite: 1]"
  },
  {
    "question": "What specific items are students required to bring with them to every lab session?[cite: 1]",
    "location_of_answer": "Under the 'Lab Format' section, located in the bulleted list directly below the phrase 'Students must bring the following to every lab session:'.[cite: 1]"
  },
  {
    "question": "Is there a grace period for being late, or a cutoff time when students will no longer be admitted?[cite: 1]",
    "location_of_answer": "Under the 'Schedule' section, located in the third bullet point regarding punctuality.[cite: 1]"
  },
  {
    "question": "What steps must a student take if they miss a lab due to illness, and what documentation is required?[cite: 1]",
    "location_of_answer": "Under the 'Requirements and Policies' section, specifically beneath the 'Attendance and Makeup Policy' subheading in the list of valid reasons for absence.[cite: 1]"
  },
  {
    "question": "How long does a student have to dispute or ask questions about a grade once it has been posted?[cite: 1]",
    "location_of_answer": "Under the 'Requirements and Policies' section, specifically beneath the 'Grade Review and Inquiry Policy' subheading in the second bullet point.[cite: 1]"
  },
  {
    "question": "Are students allowed to use their laptops or cell phones during the lab?[cite: 1]",
    "location_of_answer": "Under the 'Requirements and Policies' section, specifically beneath the 'Digital Devices & Participation Policy' subheading in the first and second bullet points.[cite: 1]"
  },
  {
    "question": "What happens if a student forgets to print and bring their lab manual?[cite: 1]",
    "location_of_answer": "Under the 'Requirements and Policies' section, specifically beneath the 'Digital Devices & Participation Policy' subheading in the final long bullet point.[cite: 1]"
  }
]

[
  {
    "question": "What materials are students required to bring to every tutorial session?[cite: 1]",
    "answer_location": "In the 'Introduction' section, under the 'Preparation and Materials' subheading.[cite: 1]"
  },
  {
    "question": "Are students allowed to use cell phones, laptops, or other smart devices during the session?[cite: 1]",
    "answer_location": "In the 'Introduction' section, under the 'Digital Tools Policy' subheading.[cite: 1]"
  },
  {
    "question": "What is the proper protocol and email address for communicating with the instructor?[cite: 1]",
    "answer_location": "In the 'Introduction' section, under the 'Communication' subheading.[cite: 1]"
  },
  {
    "question": "When and where are the instructor's office hours held?[cite: 1]",
    "answer_location": "In the 'Tutorial Policies and Procedures' section, under the 'Office Hours' subheading.[cite: 1]"
  },
  {
    "question": "What specific resources are permitted during the open-book written tutorial reports?[cite: 1]",
    "answer_location": "In the 'Tutorial Policies and Procedures' section, under the 'Tutorial Format' subheading, specifically within the 'Permitted Resources' bulleted list.[cite: 1]"
  },
  {
    "question": "How much time do students have to raise concerns about their grades after a tutorial has been marked?[cite: 1]",
    "answer_location": "In the 'Tutorial Policies and Procedures' section, under the 'Attendance and Grading Policies' subheading, specifically at numbered point 18.[cite: 1]"
  },
  {
    "question": "What is the penalty for using unauthorized digital tools or AI-based systems during a tutorial?[cite: 1]",
    "answer_location": "In the 'Tutorial Policies and Procedures' section, under the 'Tutorial Format' subheading, in the paragraph immediately following the 'Permitted Resources' list.[cite: 1]"
  }
]

[{'question': 'What materials are students required to bring to every tutorial session?[cite: 1]',
  'answer_location': "In the 'Introduction' section, under the 'Preparation and Materials' subheading.[cite: 1]"},
 {'question': 'Are students allowed to use cell phones, laptops, or other smart devices during the session?[cite: 1]',
  'answer_location': "In the 'Introduction' section, under the 'Digital Tools Policy' subheading.[cite: 1]"},
 {'question': 'What is the proper protocol and email address for communicating with the instructor?[cite: 1]',
  'answer_location': "In the 'Introduction' section, under the 'Communication' subheading.[cite: 1]"},
 {'question': "When and where are the instructor's office hours held?[cite: 1]",
  'answer_location': "In the 'Tutorial Policies and Procedures' section, under the 'Office Hours' subheading.[cite: 1]"},
 {'question': 'What specific resources are permitted during the open-book written tutorial reports?[cite: 1]',
  'answer_location': "In the 

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("API Key loaded successfully.")
except Exception:
    print("Add 'GOOGLE_API_KEY' to your Colab notebook Secrets (key icon in the left sidebar).")

API Key loaded successfully.


In [ ]:
# =====================================================================
# CELL 6: Execution Pipeline (Main)
# =====================================================================
if __name__ == "__main__":
    # 1. Initialize the Gemma GenAI model client
    # If you have an API key set in your environment as GOOGLE_API_KEY, it will be detected automatically.
    # Otherwise, you can pass api_key="YOUR_KEY" here.
    gemma_client = GemmaGenAIModel()

    # Define files available in the working directory
    pdf_files = [
        "PC141labRules.pdf"
    ]

    # Execute Pipeline Steps
    print("Chunking documents...")
    chunks = load_and_chunk_documents(pdf_files)

    print("Applying LLM Semantic Context Tags...")
    tagged_chunks = tag_chunks_with_llm(chunks, model_client=gemma_client)

    print("Indexing into Hybrid Vector/BM25 Database...")
    hybrid_retriever = HybridRetriever(tagged_chunks)

    # Test Query
    test_query = "What is the penalty for a late lab submission and can I use my phone?"
    print(f"\nQuery: {test_query}\n")

    final_answer = answer_query(test_query, hybrid_retriever, model_client=gemma_client)
    print("System Response:\n", final_answer)

Chunking documents...
Applying LLM Semantic Context Tags...
Indexing into Hybrid Vector/BM25 Database...

Query: What is the penalty for a late lab submission and can I use my phone?



AssertionError: Unknown arguments: ['query_vector']